<a href="https://colab.research.google.com/github/aaronseymour7/MLQ/blob/main/uCJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1D Heisenberg: Re-uCJ vs g-uCJ

This notebook implements and compares two variants of the unitary cluster Jastrow (uCJ) ansatz
for the 1D Heisenberg model:

- **Re-uCJ** — real orbital rotations only (Givens rotations with real angles)
- **g-uCJ** — generalized uCJ with both real (`K_real`) and imaginary (`K_imag`) orbital rotation components

Both variants use an RBM warm start via NetKet VMC.

**Sections:**
1. Install & Imports
2. Global Configuration
3. Shared Utilities (Hamiltonian, JAX helpers, NetKet VMC, Primitives)
4. Re-uCJ
5. g-uCJ
6. Comparison Runner

## 1. Install & Imports

In [ ]:
!pip install netket
!pip install qiskit
!pip install --upgrade numba netket
!pip install --upgrade pennylane
!pip install ffsim
!pip install qiskit-aer
!pip install qiskit-algorithms

In [ ]:
import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import netket as nk
import netket.nn as nknn
import flax.linen as nn
import jax.numpy as jnp
import jax
import optax

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp

## 2. Global Configuration

In [ ]:
J_COUPLING  = 1.0
PBC         = True
ALPHA       = 3
VMC_SAMPLES = 1024
VMC_STEPS   = 600
K_LAYERS    = 3
K_MAX       = 3
E_TOL       = 5e-3
SEED        = 23
N_RESTARTS  = 3

jax.config.update("jax_enable_x64", True)

## 3. Shared Utilities

### 3a. Exact Diagonalization & Hilbert Space

In [ ]:
def build_basis(n, n_up):
    return np.array([b for b in range(1 << n) if bin(b).count('1') == n_up],
                    dtype=np.int64)


def build_hamiltonian(n, n_up, j=1.0, pbc=True):
    basis   = build_basis(n, n_up)
    dim     = len(basis)
    idx_map = {int(b): i for i, b in enumerate(basis)}
    H       = lil_matrix((dim, dim), dtype=np.float64)
    edges   = ([(i, (i+1) % n) for i in range(n)] if pbc
               else [(i, i+1) for i in range(n-1)])

    for si, sj in edges:
        for row, bits in enumerate(basis):
            zi = 0.5 if (bits >> si) & 1 else -0.5
            zj = 0.5 if (bits >> sj) & 1 else -0.5
            H[row, row] += j * zi * zj                   # Sz·Sz
            if ((bits >> si) & 1) != ((bits >> sj) & 1): # S+S- + S-S+
                flipped = bits ^ (1 << si) ^ (1 << sj)
                col = idx_map.get(int(flipped), -1)
                if col >= 0:
                    H[row, col] += 0.5 * j
    return csr_matrix(H), basis, idx_map

### 3b. JAX Hamiltonian & State Utilities

In [ ]:
def build_jax_hamiltonian(n, n_up, j=1.0, pbc=True):
    """
    Returns (H_rows, H_cols, H_vals) as JAX int/float arrays.
    H|ψ⟩ = scatter_add(H_vals * ψ[H_cols], H_rows, dim=DIM)
    """
    basis_list = [b for b in range(1<<n) if bin(b).count('1')==n_up]
    idx_map    = {b:i for i,b in enumerate(basis_list)}
    rows, cols, vals = [], [], []
    edges = [(i,(i+1)%n) for i in range(n)] if pbc else [(i,i+1) for i in range(n-1)]
    for i, j_site in edges:
        for row, bits in enumerate(basis_list):
            zi = 0.5 if (bits>>i)&1 else -0.5
            zj = 0.5 if (bits>>j_site)&1 else -0.5
            rows.append(row); cols.append(row); vals.append(j * zi * zj)
            if ((bits>>i)&1) != ((bits>>j_site)&1):
                fl = bits ^ (1<<i) ^ (1<<j_site)
                if fl in idx_map:
                    rows.append(row); cols.append(idx_map[fl]); vals.append(0.5*j)
    return (jnp.array(rows, dtype=jnp.int32),
            jnp.array(cols, dtype=jnp.int32),
            jnp.array(vals, dtype=jnp.float64))


def make_apply_H_jax(h_rows, h_cols, h_vals, dim):
    """Returns a JIT-compiled H|ψ⟩ function for the given sparse H."""
    @jax.jit
    def apply_H(psi):
        return jnp.zeros(dim, dtype=psi.dtype).at[h_rows].add(h_vals * psi[h_cols])
    return apply_H


def neel_jax(n, n_up, basis, idx_map):
    """
    Néel state in the Sz sector: even sites occupied (|1⟩=↓).
    For odd N this gives ceil(N/2) = N_UP up-spins. ✓
    """
    neel_bits = sum(1<<i for i in range(n) if i%2==0)
    assert neel_bits in idx_map, f"Neel state not in Sz sector (N={n}, N_up={n_up})"
    psi = jnp.zeros(len(basis), dtype=jnp.complex128)
    return psi.at[idx_map[neel_bits]].set(1.0)


def build_jastrow_matrix(n, basis):
    """
    Returns NN_MAT of shape (DIM, n_pair) where NN_MAT[state, k] = n_i*n_j
    for the k-th pair (i,j). Used as: phases = NN_MAT @ theta_J
    """
    n_pair = n*(n-1)//2
    mat    = np.zeros((len(basis), n_pair))
    k = 0
    for i in range(n):
        for j in range(i+1, n):
            for row, bits in enumerate(basis):
                mat[row, k] = ((bits>>i)&1) * ((bits>>j)&1)
            k += 1
    return jnp.array(mat, dtype=jnp.float64)


def build_givens_pairs(n, basis, idx_map):
    """
    For each (i,j) pair, return (srcs, dsts) index arrays for the
    states where site i is occupied and site j is empty.
    These are static — only the angle theta changes.
    """
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            srcs, dsts = [], []
            for row, bits in enumerate(basis):
                if (bits>>i)&1 and not (bits>>j)&1:
                    fl = bits ^ (1<<i) ^ (1<<j)
                    if fl in idx_map:
                        srcs.append(row)
                        dsts.append(idx_map[fl])
            if srcs:
                pairs.append((jnp.array(srcs, dtype=jnp.int32),
                               jnp.array(dsts, dtype=jnp.int32)))
            else:
                pairs.append((jnp.array([], dtype=jnp.int32),
                               jnp.array([], dtype=jnp.int32)))
    return pairs

### 3c. NetKet VMC & RBM Warm Start

In [ ]:
class RBMModel(nn.Module):
    alpha: int = 1

    @nn.compact
    def __call__(self, x):
        x   = x.astype(jnp.complex128)
        a   = self.param('visible_bias', nn.initializers.normal(0.01),
                         (x.shape[-1],), jnp.complex128)
        W   = nn.Dense(self.alpha * x.shape[-1],
                       use_bias=True,
                       dtype=jnp.complex128,
                       param_dtype=jnp.complex128,
                       kernel_init=nn.initializers.normal(0.01),
                       bias_init=nn.initializers.normal(0.01))
        pre = W(x)
        return jnp.dot(x, a) + jnp.sum(nknn.log_cosh(pre), axis=-1)


def run_netket_vmc(n, e_exact, j=1.0, pbc=True, alpha=1,
                   n_samples=512, n_steps=600, total_sz=0):
    hi  = nk.hilbert.Spin(s=0.5, N=n, total_sz=total_sz)
    # NetKet Heisenberg uses σ·σ (Pauli), not S·S (spin operators).
    # Since S = σ/2, we have S·S = σ·σ/4, so NetKet's J_nk = J/4
    # to match Lanczos where H = J sum S·S.
    ha  = nk.operator.Heisenberg(hilbert=hi,
                                  graph=nk.graph.Chain(n, pbc=pbc),
                                  J=j/4.0)
    sa  = nk.sampler.MetropolisExchange(hi, n_chains=16,
                                         graph=nk.graph.Chain(n))
    vs  = nk.vqs.MCState(sa, RBMModel(alpha=alpha),
                          n_samples=n_samples, seed=SEED)
    opt = optax.sgd(learning_rate=0.02)
    gs = nk.driver.VMC_SR(hamiltonian=ha, optimizer=opt, diag_shift=0.01, variational_state=vs)

    print(f"\n[NetKet VMC]  {n_steps} SR steps  "
          f"(N={n}, alpha={alpha}, samples={n_samples}, Sz={total_sz}/2)")
    gs.run(n_iter=n_steps, out=nk.logging.RuntimeLog())

    E_rbm = float(np.real(vs.expect(ha).mean))
    print(f"[NetKet VMC]  ⟨E⟩_RBM = {E_rbm:.8f}   E_exact = {e_exact:.8f}")
    ratio = E_rbm / e_exact if abs(e_exact) > 1e-10 else float('inf')
    if abs(ratio - 1.0) > 0.3:
        print(f"  WARNING: RBM/exact ratio = {ratio:.3f} — possible scale mismatch.")
    return vs, ha


def extract_rbm_correlators(vs, n, basis, idx_map, use_exact_basis=True):
    """
    Compute θ_J (Jastrow seed) and θ_K (orbital rotation seed) from RBM.

    use_exact_basis=True  : exact sum over full Sz sector  (default, N<=20)
    use_exact_basis=False : MC estimator fallback for very large N
    """

    # sigma must be shape (batch, N) with values in {-1, +1} (NetKet convention).
    def log_psi(sigma_batch):
        """Evaluate log ψ(σ) for a batch of configurations. Returns (batch,) complex."""
        return np.array(vs.log_value(jnp.array(sigma_batch, dtype=jnp.float32)))

    if use_exact_basis:
        # Full Sz-sector basis: (DIM, N), values ±1
        basis_arr = np.array(
            [[(1 if (b >> s) & 1 else -1) for s in range(n)] for b in basis],
            dtype=np.float32
        )   # (DIM, N)

        # All amplitudes in one call
        log_amps = log_psi(basis_arr)                             # (DIM,) complex
        # Normalise in log-space for numerical stability
        log_amps = log_amps - np.max(np.real(log_amps))
        amps     = np.exp(log_amps)                               # (DIM,) complex
        probs    = np.abs(amps) ** 2
        probs   /= probs.sum()                                    # normalise

        # ── Two-body correlators ───────────────────────────────────────────
        occ     = (basis_arr + 1) / 2                             # (DIM, N), 0/1
        n_mean  = (probs[:, None] * occ).sum(0)                   # (N,)
        nn_mean = np.einsum('d,di,dj->ij', probs, occ, occ)       # (N,N)
        C       = nn_mean - np.outer(n_mean, n_mean)

        # ── One-body density matrix ρ[i,j] = ⟨c†_i c_j⟩ ─────────────────
        # ρ[i,j] = Σ_σ |ψ(σ)|² · [ψ(σ^{j→i}) / ψ(σ)] · JW_sign(σ,i,j)
        # σ^{j→i}: annihilate j, create i  (only valid when i occ, j empty)
        rho = np.diag(n_mean.astype(complex))

        for i in range(n):
            for j in range(i + 1, n):
                mask = (basis_arr[:, i] == 1) & (basis_arr[:, j] == -1)
                if not mask.any():
                    continue

                sigma_v = basis_arr[mask]                         # (K, N)
                sigma_f = sigma_v.copy()
                sigma_f[:, i] = -1
                sigma_f[:, j] =  1

                # Single batched call for all valid configs (FIX A)
                lp_f = log_psi(sigma_f)                           # (K,) complex
                lp_v = log_psi(sigma_v)                           # (K,) complex
                ratio = np.exp(lp_f - lp_v)                       # ψ(flip)/ψ(σ)

                # Jordan-Wigner sign: parity of occupied sites strictly between i,j
                lo, hi = i, j
                jw_signs = np.array([
                    (-1) ** int(((sigma_v[k, lo+1:hi] + 1) / 2).sum())
                    for k in range(sigma_v.shape[0])
                ])

                rho[i, j] = (probs[mask] * jw_signs * ratio).sum()
                rho[j, i] = np.conj(rho[i, j])

    else:
        # MC fallback for large N
        vs.n_samples = 4096
        samples = np.array(vs.samples).reshape(-1, n)             # (S, N), ±1
        occ     = (samples + 1) / 2
        n_mean  = occ.mean(0)
        nn_mean = np.einsum('si,sj->ij', occ, occ) / len(samples)
        C       = nn_mean - np.outer(n_mean, n_mean)
        rho     = np.diag(n_mean.astype(complex))

        for i in range(n):
            for j in range(i + 1, n):
                mask = (samples[:, i] == 1) & (samples[:, j] == -1)
                if not mask.any():
                    continue
                sigma_v = samples[mask].astype(np.float32)
                sigma_f = sigma_v.copy()
                sigma_f[:, i] = -1;  sigma_f[:, j] = 1
                ratio = np.exp(log_psi(sigma_f) - log_psi(sigma_v))
                lo, hi = i, j
                jw_signs = np.array([
                    (-1) ** int(((sigma_v[k, lo+1:hi] + 1) / 2).sum())
                    for k in range(len(sigma_v))
                ])
                rho[i, j] = np.mean(jw_signs * ratio)
                rho[j, i] = np.conj(rho[i, j])


    alpha_J = 0.5
    alpha_K = 1.0

    theta_J = alpha_J * np.real(C)

    K_gen = 1j * rho

    # Decompose into circuit components
    K_real = alpha_K * np.real(K_gen)   # antisymmetric part
    K_imag = alpha_K * np.imag(K_gen)   # symmetric part
    print(f"[Warm start g-uCJ]  "
          f"|θ_J| max={np.max(np.abs(theta_J)):.4f}  "
          f"|K_real| max={np.max(np.abs(K_real)):.4f}  "
          f"|K_imag| max={np.max(np.abs(K_imag)):.4f}")

    return theta_J, K_real, K_imag

### 3d. Quantum Gate Primitives (Jastrow, Givens)

In [ ]:
def apply_jastrow_jax(psi, theta_J, nn_mat):
    """exp(i J)|ψ⟩: multiply each amplitude by exp(i * NN_MAT @ theta_J)."""
    phases = nn_mat @ theta_J
    return psi * jnp.exp(1j * phases)


def apply_givens_jax(psi, theta, srcs, dsts):
    """
    Number-conserving Givens rotation in the Sz sector.
    G(θ): psi[srcs] →  cos(θ)*psi[srcs] - sin(θ)*psi[dsts]
          psi[dsts] →  sin(θ)*psi[srcs] + cos(θ)*psi[dsts]
    """
    if len(srcs) == 0:
        return psi
    c, s    = jnp.cos(theta), jnp.sin(theta)
    p_s     = psi[srcs]
    p_d     = psi[dsts]
    return psi.at[srcs].set(c*p_s - s*p_d).at[dsts].set(s*p_s + c*p_d)


def apply_givens_imag_jax(psi, theta, srcs, dsts):
    """
    Implements exp[i θ (c†_i c_j + c†_j c_i)]
    """
    if len(srcs) == 0:
        return psi

    c, s = jnp.cos(theta), jnp.sin(theta)
    p_s  = psi[srcs]
    p_d  = psi[dsts]

    return (
        psi.at[srcs].set(c*p_s - 1j*s*p_d)
           .at[dsts].set(c*p_d - 1j*s*p_s)
    )

### 3e. Qiskit Circuit Helpers

In [ ]:
def add_neel_reference(qc, qreg, n):
    for i in range(n):
        if i % 2 == 0:
            qc.x(qreg[i])
    return qc


def _givens_gate(qc, qi, qj, theta):
    qc.cx(qj, qi)
    qc.ry(theta, qi)
    qc.cx(qi, qj)
    qc.ry(-theta, qi)
    qc.cx(qj, qi)

---
## 4. Re-uCJ

Real unitary cluster Jastrow: each layer has `n_pair` Jastrow params + `n_pair` real Givens angles.
Parameters per layer: `2 * n_pair`.

In [ ]:
# ── Re-uCJ: state ansatz ──────────────────────────────────────────────────────

def ucj_state_jax(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat):
    """Apply k_layers of (Jastrow + real Givens) to psi0. [Re-uCJ]"""
    psi = psi0
    for layer in range(k_layers):
        offset = layer * 2 * n_pair
        tJ = theta[offset           : offset + n_pair]
        tK = theta[offset + n_pair  : offset + 2*n_pair]
        psi = apply_jastrow_jax(psi, tJ, nn_mat)
        for k, (srcs, dsts) in enumerate(givens_pairs):
            psi = apply_givens_jax(psi, tK[k], srcs, dsts)
    return psi


def energy_jax(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat, apply_H):
    psi    = ucj_state_jax(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat)
    norm   = jnp.dot(jnp.conj(psi), psi)
    return jnp.real(jnp.dot(jnp.conj(psi), apply_H(psi)) / norm)


def make_jax_energy_and_grad_re(n, k_layers, psi0, givens_pairs, nn_mat, apply_H):
    import functools
    n_pair = n * (n-1) // 2
    _efn = functools.partial(
        energy_jax,
        k_layers=k_layers,
        psi0=psi0,
        n_pair=n_pair,
        givens_pairs=givens_pairs,
        nn_mat=nn_mat,
        apply_H=apply_H
    )
    _vg = jax.jit(jax.value_and_grad(_efn))
    _e  = jax.jit(_efn)
    return _vg, _e


def compute_fidelity_jax_re(theta, k_layers, psi_neel, psi_exact,
                              givens_pairs, nn_mat, n):
    """|⟨ψ_exact|Ψ(θ)⟩|² computed in the Sz sector. [Re-uCJ]"""
    n_pair = n * (n-1) // 2
    psi    = ucj_state_jax(theta, k_layers, psi_neel, n_pair, givens_pairs, nn_mat)
    return float(jnp.abs(jnp.dot(jnp.conj(psi_exact.astype(jnp.complex128)), psi))**2)

In [ ]:
# ── Re-uCJ: warm start & optimization ────────────────────────────────────────

def build_warm_start_re(theta_J_mat, theta_K_mat, n, k_layers, seed=SEED):
    """
    Build initial parameter vector.

    theta_J: seeded from RBM connected correlator — good warm start.

    theta_K: the RBM gives Im(rho_ij)=0 for real-weight models, so
             theta_K_warm=0 exactly. The Neel state is a STATIONARY POINT
             of all Givens rotations at theta=0 (by particle-hole symmetry),
             so starting at theta_K=0 gives zero gradient and L-BFGS
             declares convergence immediately without moving.
             Fix: initialise theta_K with random uniform ~[-pi/4, pi/4]
             to break the symmetry and give nonzero gradients.
             The magnitude ~pi/4 is chosen to move well away from the
             degenerate point without overshooting the landscape.
    """
    n_pair = n * (n - 1) // 2
    rng    = np.random.default_rng(seed)

    def flatten_upper(mat):
        return np.array([mat[i, j]
                         for i in range(n) for j in range(i+1, n)])

    J_flat = flatten_upper(theta_J_mat)           # from RBM: good seed
    # K: ignore near-zero RBM value, use symmetry-breaking random init
    K_flat = rng.uniform(-np.pi/4, np.pi/4, n_pair)

    out = []
    for layer in range(k_layers):
        if layer == 0:
            out.append(J_flat)
            out.append(K_flat)
        else:
            out.append(0.05 * rng.standard_normal(n_pair))   # J
            out.append(rng.uniform(-np.pi/4, np.pi/4, n_pair))  # K
    return np.concatenate(out)


def optimize_layer_re(n, k, x0, e_exact, psi_neel, psi_exact,
                      givens_pairs, nn_mat, apply_H, maxiter=600):
    """
    Minimize ⟨H⟩ using JAX autodiff + scipy L-BFGS-B. [Re-uCJ]
    val_and_grad computes energy and gradient in one forward+backward pass.
    Cost per L-BFGS step: O(2^N * N)  independent of n_params.
    """
    from scipy.optimize import minimize as scipy_minimize
    print(f"\n[Re-uCJ k={k}]  {len(x0)} params,  maxiter={maxiter}")

    val_grad_fn, energy_fn = make_jax_energy_and_grad_re(
        n, k, psi_neel, givens_pairs, nn_mat, apply_H
    )

    # Warm up JIT compilation
    _ = val_grad_fn(jnp.array(x0, dtype=jnp.float64))

    best   = [np.inf]
    it     = [0]

    def scipy_fn(x_np):
        it[0] += 1
        x     = jnp.array(x_np, dtype=jnp.float64)
        E, g  = val_grad_fn(x)
        E_f   = float(E)
        if E_f < best[0]:
            best[0] = E_f
        if it[0] % 10 == 0:
            print(f"    iter {it[0]:4d}  E={E_f:.8f}"
                  f"  best={best[0]:.8f}  exact={e_exact:.8f}"
                  f"  ΔE={E_f-e_exact:+.2e}")
        return E_f, np.array(g, dtype=np.float64)

    E0 = float(energy_fn(jnp.array(x0, dtype=jnp.float64)))
    print(f"  x0 energy: {E0:.8f}")

    result   = scipy_minimize(scipy_fn, x0, jac=True, method='L-BFGS-B',
                               options={'maxiter': maxiter,
                                        'ftol': 1e-14, 'gtol': 1e-8})
    opt_x    = np.array(result.x)
    opt_E    = float(result.fun)
    fidelity = compute_fidelity_jax_re(
        jnp.array(opt_x), k, psi_neel, psi_exact, givens_pairs, nn_mat, n
    )

    print(f" E={opt_E:.8f}  |⟨exact|uCJ⟩|²={fidelity:.6f}"
          f"  converged={result.success}  nit={result.nit}")
    return opt_x, opt_E, fidelity


def adaptive_ucj_re(n, k_init, k_max, e_tol, theta_J_warm, theta_K_warm,
                    e_exact, psi_neel, psi_exact, givens_pairs, nn_mat, apply_H,
                    maxiter=300, n_restarts=N_RESTARTS):
    """
    Adaptive k with multi-restart per layer. [Re-uCJ]
    Multi-restart is essential because the Neel→uCJ landscape has many
    local minima depending on the random K initialisation direction.
    """
    best = dict(E=np.inf, fid=0., params=None, k=k_init)
    prev_params = None

    for k in range(k_init, k_max + 1):
        n_pair = n * (n - 1) // 2
        layer_best = dict(E=np.inf, params=None, fid=0.)

        for restart in range(n_restarts):
            print(f"\n  [N={n}, k={k}, restart={restart+1}/{n_restarts}]")
            if prev_params is None:
                x0 = build_warm_start_re(theta_J_warm, theta_K_warm, n, k,
                                         seed=SEED + restart)
            else:
                # Append a new symmetry-breaking layer to previous optimised params
                rng       = np.random.default_rng(SEED + k * 100 + restart)
                new_layer = np.concatenate([
                    0.05 * rng.standard_normal(n_pair),          # J
                    rng.uniform(-np.pi/4, np.pi/4, n_pair)       # K
                ])
                x0 = np.concatenate([prev_params, new_layer])

            opt_x, opt_E, fid = optimize_layer_re(
                n, k, x0, e_exact, psi_neel, psi_exact,
                givens_pairs, nn_mat, apply_H, maxiter
            )

            if opt_E < layer_best['E']:
                layer_best.update(E=opt_E, params=opt_x, fid=fid)

            # if this restart converged, no need to try more
            if abs(opt_E - e_exact) < e_tol:
                break

        # Keep best result from all restarts for this layer
        opt_x, opt_E, fid = layer_best['params'], layer_best['E'], layer_best['fid']

        # plateau check: if best is stuck, reset prev_params so next k
        # gets a fresh RBM warm start rather than inheriting bad geometry
        _, energy_fn = make_jax_energy_and_grad_re(
            n, k, psi_neel, givens_pairs, nn_mat, apply_H
        )
        grad_norm = float(jnp.linalg.norm(
            jax.grad(energy_fn)(jnp.array(opt_x, dtype=jnp.float64))
        ))
        on_plateau = grad_norm < 1e-6 and abs(opt_E - e_exact) > e_tol
        if on_plateau:
            print(f"  WARNING: k={k} best is plateau-stuck "
                  f"(|∇E|={grad_norm:.2e}) — resetting prev_params to None.")
            prev_params = None
        else:
            prev_params = opt_x

        if opt_E < best['E']:
            best.update(E=opt_E, fid=fid, params=opt_x, k=k)

        delta = abs(opt_E - e_exact)
        print(f"\n  [Re-uCJ k={k}]  best E={opt_E:.8f}  |ΔE|={delta:.4e}  (tol={e_tol:.4e})")
        if delta < e_tol:
            print(f"  Converged at k={k}.")
            break
        elif k < k_max:
            print(f"  Adding layer {k+1}…")

    return best


def build_visualization_circuit_re(n, k_layers, opt_theta):
    """
    Build a Qiskit circuit with the optimized parameters bound in,
    purely for visualization / export. [Re-uCJ]
    """
    n_pair  = n * (n-1) // 2
    qreg    = QuantumRegister(n, 'q')
    qc      = QuantumCircuit(qreg)
    add_neel_reference(qc, qreg, n)
    for l in range(k_layers):
        qc.barrier(label=f'L{l}')
        tJ = opt_theta[l*2*n_pair       : l*2*n_pair + n_pair]
        tK = opt_theta[l*2*n_pair+n_pair : (l+1)*2*n_pair]
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                gamma = float(tJ[k]) / 4.0
                qc.cx(qreg[i], qreg[j])
                qc.rz(2.0*gamma, qreg[j])
                qc.cx(qreg[i], qreg[j])
                k += 1
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                _givens_gate(qc, qreg[i], qreg[j], float(tK[k]))
                k += 1
    return qc

In [ ]:
# ── Re-uCJ: scaling analysis ──────────────────────────────────────────────────

def scaling_analysis_vs_N_re(N_list, k_init=1, k_max=3, e_tol=1e-3,
                              maxiter=300, n_restarts=2):
    """
    For each N in N_list:
        • Compute exact ground state
        • Run adaptive Re-uCJ
        • Compute fidelity
        • Extract circuit depth
    Returns results dict and plots Energy Error, Fidelity, Circuit Depth vs N.
    """
    results = {"N": [], "E_exact": [], "E_ucj": [], "DeltaE": [],
               "Fidelity": [], "Depth": [], "k_opt": []}

    for N_val in N_list:
        print("\n" + "="*70)
        print(f"[Re-uCJ] Scaling run: N = {N_val}")
        print("="*70)

        n_up            = N_val // 2 if N_val % 2 == 0 else (N_val + 1) // 2
        netket_total_sz = 0 if N_val % 2 == 0 else 1

        H_sparse, basis_loc, bindex_loc = build_hamiltonian(N_val, n_up, J_COUPLING, PBC)
        evals, evecs = eigsh(H_sparse, k=1, which='SA')
        e_exact  = float(evals[0])
        psi_exact = evecs[:, 0] / np.linalg.norm(evecs[:, 0])
        print(f"[Lanczos]  E_exact = {e_exact:.8f}   E/site = {e_exact/N_val:.8f}")

        vs_rbm, _ = run_netket_vmc(N_val, e_exact, J_COUPLING, PBC, ALPHA,
                                   VMC_SAMPLES, VMC_STEPS, netket_total_sz)
        theta_J_warm, K_real_warm, K_imag_warm = extract_rbm_correlators(
            vs_rbm, N_val, basis_loc, bindex_loc
        )

        h_rows, h_cols, h_vals = build_jax_hamiltonian(N_val, n_up, J_COUPLING, PBC)
        dim_jax      = len(basis_loc)
        apply_H      = make_apply_H_jax(h_rows, h_cols, h_vals, dim_jax)
        givens_pairs = build_givens_pairs(N_val, list(basis_loc), bindex_loc)
        nn_mat       = build_jastrow_matrix(N_val, list(basis_loc))
        psi_neel     = neel_jax(N_val, n_up, list(basis_loc), bindex_loc)
        psi_exact_jax = jnp.array(psi_exact, dtype=jnp.complex128)

        best = adaptive_ucj_re(
            N_val, k_init, k_max, e_tol,
            theta_J_warm, K_real_warm,
            e_exact, psi_neel, psi_exact_jax,
            givens_pairs, nn_mat, apply_H,
            maxiter=maxiter, n_restarts=n_restarts
        )

        qc_vis = build_visualization_circuit_re(N_val, best['k'], jnp.array(best['params']))
        depth  = qc_vis.depth()

        results["N"].append(N_val)
        results["E_exact"].append(e_exact)
        results["E_ucj"].append(best["E"])
        results["DeltaE"].append(abs(best["E"] - e_exact))
        results["Fidelity"].append(best["fid"])
        results["Depth"].append(depth)
        results["k_opt"].append(best["k"])

    for key in results:
        results[key] = np.array(results[key])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Re-uCJ — 1D Heisenberg", fontsize=14)

    axes[0].plot(results["N"], results["DeltaE"], 'o-', color='green')
    axes[0].set_yscale("log")
    axes[0].set_xlabel("N"); axes[0].set_ylabel("|E_uCJ - E_exact|")
    axes[0].set_title("Energy Error vs N"); axes[0].grid(True)

    axes[1].plot(results["N"], results["Fidelity"], 'o-', color='blue')
    axes[1].set_xlabel("N"); axes[1].set_ylabel("Fidelity")
    axes[1].set_title("Fidelity vs N"); axes[1].grid(True)

    axes[2].plot(results["N"], results["Depth"], '.-')
    axes[2].set_xlabel("N"); axes[2].set_ylabel("Circuit Depth")
    axes[2].set_title("Circuit Depth vs N"); axes[2].grid(True)

    plt.tight_layout()
    plt.show()

    return results

---
## 5. g-uCJ

Generalized uCJ: each layer has `n_pair` Jastrow + `n_pair` real Givens + `n_pair` imaginary Givens.
Parameters per layer: `3 * n_pair`.

In [ ]:
# ── g-uCJ: state ansatz ───────────────────────────────────────────────────────

def ucj_state_jax_g(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat):
    """Apply k_layers of (Jastrow + real Givens + imag Givens) to psi0. [g-uCJ]"""
    psi = psi0
    for layer in range(k_layers):
        offset = layer * 3 * n_pair
        tJ     = theta[offset             : offset + n_pair]
        tK_r   = theta[offset + n_pair    : offset + 2*n_pair]
        tK_i   = theta[offset + 2*n_pair  : offset + 3*n_pair]
        psi = apply_jastrow_jax(psi, tJ, nn_mat)
        for k, (srcs, dsts) in enumerate(givens_pairs):
            psi = apply_givens_jax(psi, tK_r[k], srcs, dsts)
        for k, (srcs, dsts) in enumerate(givens_pairs):
            psi = apply_givens_imag_jax(psi, tK_i[k], srcs, dsts)
    return psi


def energy_jax_g(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat, apply_H):
    psi  = ucj_state_jax_g(theta, k_layers, psi0, n_pair, givens_pairs, nn_mat)
    norm = jnp.dot(jnp.conj(psi), psi)
    return jnp.real(jnp.dot(jnp.conj(psi), apply_H(psi)) / norm)


def make_jax_energy_and_grad_g(n, k_layers, psi0, givens_pairs, nn_mat, apply_H):
    import functools
    n_pair = n * (n-1) // 2
    _efn = functools.partial(
        energy_jax_g,
        k_layers=k_layers,
        psi0=psi0,
        n_pair=n_pair,
        givens_pairs=givens_pairs,
        nn_mat=nn_mat,
        apply_H=apply_H
    )
    _vg = jax.jit(jax.value_and_grad(_efn))
    _e  = jax.jit(_efn)
    return _vg, _e


def compute_fidelity_jax_g(theta, k_layers, psi_neel, psi_exact,
                             givens_pairs, nn_mat, n):
    """|⟨ψ_exact|Ψ(θ)⟩|² computed in the Sz sector. [g-uCJ]"""
    n_pair = n * (n-1) // 2
    psi    = ucj_state_jax_g(theta, k_layers, psi_neel, n_pair, givens_pairs, nn_mat)
    return float(jnp.abs(jnp.dot(jnp.conj(psi_exact.astype(jnp.complex128)), psi))**2)

In [ ]:
# ── g-uCJ: warm start & optimization ─────────────────────────────────────────

def build_warm_start_g(theta_J_mat, K_real_mat, K_imag_mat,
                       n, k_layers, seed=SEED):

    n_pair = n * (n - 1) // 2
    rng    = np.random.default_rng(seed)

    def flatten_upper(mat):
        return np.array([mat[i, j]
                         for i in range(n)
                         for j in range(i+1, n)])

    J_flat      = flatten_upper(theta_J_mat)
    K_real_flat = flatten_upper(K_real_mat)
    K_imag_flat = flatten_upper(K_imag_mat)

    out = []

    for layer in range(k_layers):
        if layer == 0:
            # RBM seed
            out.append(J_flat)
            out.append(K_real_flat)
            out.append(K_imag_flat)
        else:
            # random symmetry-breaking layers
            out.append(0.05 * rng.standard_normal(n_pair))
            out.append(rng.uniform(-np.pi/4, np.pi/4, n_pair))
            out.append(rng.uniform(-np.pi/4, np.pi/4, n_pair))

    return np.concatenate(out)


def optimize_layer_g(n, k, x0, e_exact, psi_neel, psi_exact,
                     givens_pairs, nn_mat, apply_H, maxiter=600):
    """
    Minimize ⟨H⟩ using JAX autodiff + scipy L-BFGS-B. [g-uCJ]
    val_and_grad computes energy and gradient in one forward+backward pass.
    Cost per L-BFGS step: O(2^N * N)  independent of n_params.
    """
    from scipy.optimize import minimize as scipy_minimize
    print(f"\n[g-uCJ k={k}]  {len(x0)} params,  maxiter={maxiter}")

    val_grad_fn, energy_fn = make_jax_energy_and_grad_g(
        n, k, psi_neel, givens_pairs, nn_mat, apply_H
    )

    # Warm up JIT compilation
    _ = val_grad_fn(jnp.array(x0, dtype=jnp.float64))

    best   = [np.inf]
    it     = [0]

    def scipy_fn(x_np):
        it[0] += 1
        x     = jnp.array(x_np, dtype=jnp.float64)
        E, g  = val_grad_fn(x)
        E_f   = float(E)
        if E_f < best[0]:
            best[0] = E_f
        if it[0] % 10 == 0:
            print(f"    iter {it[0]:4d}  E={E_f:.8f}"
                  f"  best={best[0]:.8f}  exact={e_exact:.8f}"
                  f"  ΔE={E_f-e_exact:+.2e}")
        return E_f, np.array(g, dtype=np.float64)

    E0 = float(energy_fn(jnp.array(x0, dtype=jnp.float64)))
    print(f"  x0 energy: {E0:.8f}")

    result   = scipy_minimize(scipy_fn, x0, jac=True, method='L-BFGS-B',
                               options={'maxiter': maxiter,
                                        'ftol': 1e-14, 'gtol': 1e-8})
    opt_x    = np.array(result.x)
    opt_E    = float(result.fun)
    fidelity = compute_fidelity_jax_g(
        jnp.array(opt_x), k, psi_neel, psi_exact, givens_pairs, nn_mat, n
    )

    print(f"  E={opt_E:.8f}  |⟨exact|uCJ⟩|²={fidelity:.6f}"
          f"  converged={result.success}  nit={result.nit}")
    return opt_x, opt_E, fidelity


def adaptive_ucj_g(n, k_init, k_max, e_tol, theta_J_warm, K_real_warm, K_imag_warm,
                   e_exact, psi_neel, psi_exact, givens_pairs, nn_mat, apply_H,
                   maxiter=300, n_restarts=N_RESTARTS):
    """
    Adaptive k with multi-restart per layer. [g-uCJ]
    Multi-restart is essential because the Neel→uCJ landscape has many
    local minima depending on the random K initialisation direction.
    """
    best = dict(E=np.inf, fid=0., params=None, k=k_init)
    prev_params = None

    for k in range(k_init, k_max + 1):
        n_pair = n * (n - 1) // 2
        layer_best = dict(E=np.inf, params=None, fid=0.)

        for restart in range(n_restarts):
            print(f"\n  [k={k}, restart={restart+1}/{n_restarts}]")
            if prev_params is None:
                x0 = build_warm_start_g(theta_J_warm, K_real_warm, K_imag_warm, n, k,
                                        seed=SEED + restart)
            else:
                # Append a new symmetry-breaking layer to previous optimised params
                rng       = np.random.default_rng(SEED + k * 100 + restart)
                new_layer = np.concatenate([
                    0.05 * rng.standard_normal(n_pair),          # J
                    rng.uniform(-np.pi/4, np.pi/4, n_pair),      # K_real
                    rng.uniform(-np.pi/4, np.pi/4, n_pair)       # K_imag
                ])
                x0 = np.concatenate([prev_params, new_layer])

            opt_x, opt_E, fid = optimize_layer_g(
                n, k, x0, e_exact, psi_neel, psi_exact,
                givens_pairs, nn_mat, apply_H, maxiter
            )

            if opt_E < layer_best['E']:
                layer_best.update(E=opt_E, params=opt_x, fid=fid)

        # Keep best result from all restarts for this layer
        opt_x, opt_E, fid = layer_best['params'], layer_best['E'], layer_best['fid']
        prev_params = opt_x

        if opt_E < best['E']:
            best.update(E=opt_E, fid=fid, params=opt_x, k=k)

        delta = abs(opt_E - e_exact)
        print(f"\n  [g-uCJ k={k}]  best E={opt_E:.8f}  |ΔE|={delta:.4e}  (tol={e_tol:.4e})")
        if delta < e_tol:
            print(f"  Converged at k={k}.")
            break
        elif k < k_max:
            print(f"  Adding layer {k+1}…")

    return best


def build_visualization_circuit_g(n, k_layers, opt_theta):
    """
    Build a Qiskit circuit with the optimized parameters bound in,
    purely for visualization / export. [g-uCJ]
    """
    n_pair  = n * (n-1) // 2
    qreg    = QuantumRegister(n, 'q')
    qc      = QuantumCircuit(qreg)
    add_neel_reference(qc, qreg, n)
    for l in range(k_layers):
        qc.barrier(label=f'L{l}')
        tJ   = opt_theta[l*3*n_pair            : l*3*n_pair + n_pair]
        tK   = opt_theta[l*3*n_pair + n_pair   : (l+1)*3*n_pair]
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                gamma = float(tJ[k]) / 4.0
                qc.cx(qreg[i], qreg[j])
                qc.rz(2.0*gamma, qreg[j])
                qc.cx(qreg[i], qreg[j])
                k += 1
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                _givens_gate(qc, qreg[i], qreg[j], float(tK[k]))
                k += 1
    return qc

In [ ]:
# ── g-uCJ: scaling analysis ───────────────────────────────────────────────────

def scaling_analysis_vs_N_g(N_list, k_init=1, k_max=3, e_tol=1e-3,
                             maxiter=300, n_restarts=2):
    """
    For each N in N_list:
        • Compute exact ground state
        • Run adaptive g-uCJ
        • Compute fidelity
        • Extract circuit depth
    Returns results dict and plots Energy Error, Fidelity, Circuit Depth vs N.
    """
    results = {"N": [], "E_exact": [], "E_ucj": [], "DeltaE": [],
               "Fidelity": [], "Depth": [], "k_opt": []}

    for N_val in N_list:
        print("\n" + "="*70)
        print(f"[g-uCJ] Scaling run: N = {N_val}")
        print("="*70)

        n_up            = N_val // 2 if N_val % 2 == 0 else (N_val + 1) // 2
        netket_total_sz = 0 if N_val % 2 == 0 else 1

        H_sparse, basis_loc, bindex_loc = build_hamiltonian(N_val, n_up, J_COUPLING, PBC)
        evals, evecs = eigsh(H_sparse, k=1, which='SA')
        e_exact  = float(evals[0])
        psi_exact = evecs[:, 0] / np.linalg.norm(evecs[:, 0])
        print(f"[Lanczos]  E_exact = {e_exact:.8f}   E/site = {e_exact/N_val:.8f}")

        vs_rbm, _ = run_netket_vmc(N_val, e_exact, J_COUPLING, PBC, ALPHA,
                                   VMC_SAMPLES, VMC_STEPS, netket_total_sz)
        theta_J_warm, K_real_warm, K_imag_warm = extract_rbm_correlators(
            vs_rbm, N_val, basis_loc, bindex_loc
        )

        h_rows, h_cols, h_vals = build_jax_hamiltonian(N_val, n_up, J_COUPLING, PBC)
        dim_jax      = len(basis_loc)
        apply_H      = make_apply_H_jax(h_rows, h_cols, h_vals, dim_jax)
        givens_pairs = build_givens_pairs(N_val, list(basis_loc), bindex_loc)
        nn_mat       = build_jastrow_matrix(N_val, list(basis_loc))
        psi_neel     = neel_jax(N_val, n_up, list(basis_loc), bindex_loc)
        psi_exact_jax = jnp.array(psi_exact, dtype=jnp.complex128)

        best = adaptive_ucj_g(
            N_val, k_init, k_max, e_tol,
            theta_J_warm, K_real_warm, K_imag_warm,
            e_exact, psi_neel, psi_exact_jax,
            givens_pairs, nn_mat, apply_H,
            maxiter=maxiter, n_restarts=n_restarts
        )

        qc_vis = build_visualization_circuit_g(N_val, best['k'], jnp.array(best['params']))
        depth  = qc_vis.depth()

        results["N"].append(N_val)
        results["E_exact"].append(e_exact)
        results["E_ucj"].append(best["E"])
        results["DeltaE"].append(abs(best["E"] - e_exact))
        results["Fidelity"].append(best["fid"])
        results["Depth"].append(depth)
        results["k_opt"].append(best["k"])

    for key in results:
        results[key] = np.array(results[key])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("g-uCJ — 1D Heisenberg", fontsize=14)

    axes[0].plot(results["N"], results["DeltaE"], 'o-', color='green')
    axes[0].set_yscale("log")
    axes[0].set_xlabel("N"); axes[0].set_ylabel("|E_uCJ - E_exact|")
    axes[0].set_title("Energy Error vs N"); axes[0].grid(True)

    axes[1].plot(results["N"], results["Fidelity"], 'o-', color='blue')
    axes[1].set_xlabel("N"); axes[1].set_ylabel("Fidelity")
    axes[1].set_title("Fidelity vs N"); axes[1].grid(True)

    axes[2].plot(results["N"], results["Depth"], '.-')
    axes[2].set_xlabel("N"); axes[2].set_ylabel("Circuit Depth")
    axes[2].set_title("Circuit Depth vs N"); axes[2].grid(True)

    plt.tight_layout()
    plt.show()

    return results

---
## 6. Comparison Runner

Run both methods on the same `N_LIST` and plot side-by-side comparisons.

In [ ]:
# ── Configure and run ─────────────────────────────────────────────────────────
N_LIST     = [4,6,8,10,12]   # system sizes to compare
K_INIT     = 1
MAXITER    = 300
N_REST     = N_RESTARTS

print("Running Re-uCJ ...")
res_re = scaling_analysis_vs_N_re(
    N_LIST, k_init=K_INIT, k_max=K_MAX, e_tol=E_TOL,
    maxiter=MAXITER, n_restarts=N_REST
)

print("\nRunning g-uCJ ...")
res_g = scaling_analysis_vs_N_g(
    N_LIST, k_init=K_INIT, k_max=K_MAX, e_tol=E_TOL,
    maxiter=MAXITER, n_restarts=N_REST
)

# ── Head-to-head comparison plots ────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Re-uCJ vs g-uCJ — 1D Heisenberg", fontsize=14)

# Energy error
axes[0].plot(res_re["N"], res_re["DeltaE"], 'o-', color='steelblue',  label='Re-uCJ')
axes[0].plot(res_g["N"],  res_g["DeltaE"],  's--', color='darkorange', label='g-uCJ')
axes[0].set_yscale("log")
axes[0].set_xlabel("N"); axes[0].set_ylabel("|E - E_exact|")
axes[0].set_title("Energy Error vs N")
axes[0].legend(); axes[0].grid(True)

# Fidelity
axes[1].plot(res_re["N"], res_re["Fidelity"], 'o-', color='steelblue',  label='Re-uCJ')
axes[1].plot(res_g["N"],  res_g["Fidelity"],  's--', color='darkorange', label='g-uCJ')
axes[1].set_xlabel("N"); axes[1].set_ylabel("Fidelity")
axes[1].set_title("Fidelity vs N")
axes[1].legend(); axes[1].grid(True)

# Circuit depth
axes[2].plot(res_re["N"], res_re["Depth"], 'o-', color='steelblue',  label='Re-uCJ')
axes[2].plot(res_g["N"],  res_g["Depth"],  's--', color='darkorange', label='g-uCJ')
axes[2].set_xlabel("N"); axes[2].set_ylabel("Circuit Depth")
axes[2].set_title("Circuit Depth vs N")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'N':>4}  {'E_exact':>12}  "
      f"{'Re-uCJ ΔE':>12}  {'Re Fid':>8}  {'Re k':>5}  {'Re Depth':>8}"
      f"{'g-uCJ ΔE':>12}  {'g Fid':>8}  {'g k':>5}  {'g Depth':>8}")
print("-" * 85)
for i, N in enumerate(res_re["N"]):
    print(f"{int(N):>4}  {res_re['E_exact'][i]:>12.6f}  "
          f"{res_re['DeltaE'][i]:>12.2e}  {res_re['Fidelity'][i]:>8.5f}  {int(res_re['k_opt'][i]):>5}  {int(res_re['Depth'][i]):>9} "
          f"{res_g['DeltaE'][i]:>12.2e}  {res_g['Fidelity'][i]:>8.5f}  {int(res_g['k_opt'][i]):>5}  {int(res_g['Depth'][i]):>8}")

In [ ]:
N_LIST     = [16]   # system sizes to compare
K_INIT     = 1
MAXITER    = 300
N_REST     = N_RESTARTS
'''
print("Running Re-uCJ ...")
res_re = scaling_analysis_vs_N_re(
    N_LIST, k_init=K_INIT, k_max=K_MAX, e_tol=E_TOL,
    maxiter=MAXITER, n_restarts=N_REST
)
'''
print("\nRunning g-uCJ ...")
res_g = scaling_analysis_vs_N_g(
    N_LIST, k_init=K_INIT, k_max=K_MAX, e_tol=E_TOL,
    maxiter=MAXITER, n_restarts=N_REST
)

In [ ]:
# ── Head-to-head comparison plots ────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Re-uCJ vs g-uCJ — 1D Heisenberg", fontsize=14)

# Energy error
axes[0].plot(res_re["N"], res_re["DeltaE"], 'o-', color='steelblue',  label='Re-uCJ')
axes[0].plot(res_g["N"],  res_g["DeltaE"],  's--', color='darkorange', label='g-uCJ')
axes[0].set_yscale("log")
axes[0].set_xlabel("N"); axes[0].set_ylabel("|E - E_exact|")
axes[0].set_title("Energy Error vs N")
axes[0].legend(); axes[0].grid(True)

# Fidelity
axes[1].plot(res_re["N"], res_re["Fidelity"], 'o-', color='steelblue',  label='Re-uCJ')
axes[1].plot(res_g["N"],  res_g["Fidelity"],  's--', color='darkorange', label='g-uCJ')
axes[1].set_xlabel("N"); axes[1].set_ylabel("Fidelity")
axes[1].set_title("Fidelity vs N")
axes[1].legend(); axes[1].grid(True)

# Circuit depth
axes[2].plot(res_re["N"], res_re["Depth"], 'o-', color='steelblue',  label='Re-uCJ')
axes[2].plot(res_g["N"],  res_g["Depth"],  's--', color='darkorange', label='g-uCJ')
axes[2].set_xlabel("N"); axes[2].set_ylabel("Circuit Depth")
axes[2].set_title("Circuit Depth vs N")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'N':>4}  {'E_exact':>12}  "
      f"{'Re-uCJ ΔE':>12}  {'Re Fid':>8}  {'Re k':>5}  "
      f"{'g-uCJ ΔE':>12}  {'g Fid':>8}  {'g k':>5}")
print("-" * 85)
for i, N in enumerate(res_re["N"]):
    print(f"{int(N):>4}  {res_re['E_exact'][i]:>12.6f}  "
          f"{res_re['DeltaE'][i]:>12.2e}  {res_re['Fidelity'][i]:>8.5f}  {int(res_re['k_opt'][i]):>5}  "
          f"{res_g['DeltaE'][i]:>12.2e}  {res_g['Fidelity'][i]:>8.5f}  {int(res_g['k_opt'][i]):>5}")